In [81]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from time import sleep

import pandas as pd

In [82]:
# 別途parameter_personal.pyを定義して、そこにUSERNAMEとPASSWORDを定義してください。
import parameter_personal as params
USERNAME = params.USERNAME  # ユーザー名を設定してください
PASSWORD = params.PASSWORD  # パスワードを設定してください

In [87]:
drive_path = r"C:\Users\zeroc\work\PythonTemplate\webscraping\drvier\chromedriver.exe"
service = webdriver.ChromeService(executable_path=drive_path)
options = Options()
options.add_argument("--headless")
driver = webdriver.Chrome(service=service, options=options)
target_url = 'https://alert.shop-bell.com/'

driver.get(target_url)
sleep(1)

login_page = 'https://alert.shop-bell.com/users/login/'

driver.get(login_page)
sleep(1)

username_input = driver.find_element(By.XPATH, "//input[@id='Email']")
username_input.send_keys(USERNAME)
sleep(1)

password_input = driver.find_element(By.XPATH, "//input[@id='Password']")
password_input.send_keys(PASSWORD)
sleep(1)

login_button = driver.find_element(By.XPATH, "//button[@type='submit' and @class='btn']")
login_button.click()
sleep(1)

list_page = 'https://alert.shop-bell.com/users/alert_list/'
driver.get(list_page)
sleep(1)

table = driver.find_element(By.XPATH, "//tbody")

table_data = []
for tr_element in table.find_elements(By.XPATH, ".//tr"):
    data_row = [td.text for td in tr_element.find_elements(By.XPATH, ".//td")]
    table_data.append(data_row)

df = pd.DataFrame(table_data)
# 欠損値があるものは排除
df = df.dropna(how='any')

df

,0,1,2,3,4,5,6,7
1,16296412,カテゴリー：\n本・コミック・雑誌\nタイトル：\nイマジナリー\n著者：\n幾花にいろ\n...,,2022年07月29日 発売,2巻,発売未定,3巻,編集 一覧 削除紙⇔電子 切換え
2,16296401,カテゴリー：\n本・コミック・雑誌\nタイトル：\n野原ひろし 昼メシの流儀\n著者：\n塚...,,2023年06月12日 発売,11巻,2024年05月10日 発売予定,12巻,編集 一覧 削除紙⇔電子 切換え
3,16296384,カテゴリー：\n本・コミック・雑誌\nタイトル：\nシメジ シミュレーション\n著者：\nつ...,,2024年01月26日 発売,5巻,発売未定,完結,編集 一覧 削除紙⇔電子 切換え
4,16296376,カテゴリー：\n本・コミック・雑誌\nタイトル：\n日常\n著者：\nあらゐけいいち\n販売...,,2022年12月26日 発売,11巻,(2029年12月26日頃 発売予想),12巻,編集 一覧 削除紙⇔電子 切換え
5,16296334,カテゴリー：\n本・コミック・雑誌\nタイトル：\nよふかしのうた\n著者：\nコトヤマ\n...,,2024年03月18日 発売,20巻,発売未定,完結,編集 一覧 削除紙⇔電子 切換え
6,15656140,カテゴリー：\n本・コミック・雑誌\nタイトル：\nONE PIECE\n著者：\n尾田栄一...,,2024年03月04日 発売,108巻,(2024年07月05日頃 発売予想),109巻,編集 一覧 削除紙⇔電子 切換え
7,15656114,カテゴリー：\n本・コミック・雑誌\nタイトル：\nワンナイト・モーニング\n著者：\n奥山...,,2023年12月11日 発売,10巻,2024年04月22日 発売予定,11巻,編集 一覧 削除紙⇔電子 切換え
8,15651054,カテゴリー：\n本・コミック・雑誌\nタイトル：\n喰う寝るふたり 住むふたり 続\n著者：...,,2023年08月19日 発売,5巻,発売未定,完結,編集 一覧 削除紙⇔電子 切換え
9,15566233,カテゴリー：\n本・コミック・雑誌\nタイトル：\n進撃のえろ子さん～変なお姉さんは男子高生...,,2024年02月29日 発売,7巻,(2024年12月01日頃 発売予想),8巻,編集 一覧 削除紙⇔電子 切換え
10,15561791,カテゴリー：\n本・コミック・雑誌\nタイトル：\n映像研には手を出すな!\n著者：\n大童...,,2023年07月12日 発売,8巻,(2024年07月11日頃 発売予想),9巻,編集 一覧 削除紙⇔電子 切換え


In [88]:
df_result = pd.DataFrame(columns=['title','num_turns','num_turns_next','release','release_next','status'])
df_result

,title,num_turns,num_turns_next,release,release_next,status


In [91]:
# タイトル抽出
import re
def cransed_title(title: str) -> str:
    ret = ''
    match = re.search(r'タイトル：\n(.*?)\n著者：', title)
    if match:
        ret = match.group(1)
    else:
        ret = 'Error: cransed_title'
    return ret


In [99]:
# 発売日データ型変更
rnum = df.iloc[1,3]
rnum

'2023年06月12日 発売'

In [105]:
import datetime
def change_datatime(date_str):
    date_pattern = r"(\d{4})年(\d{1,2})月(\d{1,2})日"  # Capture year, month, and day
    expected_pattern = r"\((\d{4})年(\d{1,2})月(\d{1,2})日頃\)"  # Capture expected release date
    
    # Match against the date pattern
    date_match = re.search(date_pattern, date_str)
    expected_match = re.search(expected_pattern, date_str)

    if date_match:
        year = int(date_match.group(1))
        month = int(date_match.group(2))
        day = int(date_match.group(3))

        release_date = f"{year}-{month:02}-{day:02}"

    elif expected_match:
        expected_year = int(expected_match.group(1))
        expected_month = int(expected_match.group(2))
        expected_day = int(expected_match.group(3))

        release_date = f"{expected_year}-{expected_month:02}-{expected_day:02}"

    else:
        release_date = ''
        print("Invalid date format")

    return release_date

rc = change_datatime(rnum)
rc

'2023-06-12'

In [109]:
# df_result = pd.DataFrame(columns=['title','num_turns','num_turns_next','release','release_next','status'])
# 1:title, 3:release, 4:巻数, 5:release_next, 6:巻数_next,
# Noneが入らないように注意すること
df_result['title'] = df[1].apply(cransed_title)
df_result['release'] = df[3].apply(change_datatime)
df_result['num_turns'] = df[4]
df_result['release_next'] = df[5].apply(change_datatime)
df_result['num_turns_next'] = df[6]
df_result

Invalid date format
Invalid date format
Invalid date format
Invalid date format
Invalid date format
Invalid date format
Invalid date format


,title,num_turns,num_turns_next,release,release_next,status
1,イマジナリー,2巻,3巻,2022-07-29,,NaN
2,野原ひろし 昼メシの流儀,11巻,12巻,2023-06-12,2024-05-10,NaN
3,シメジ シミュレーション,5巻,完結,2024-01-26,,NaN
4,日常,11巻,12巻,2022-12-26,2029-12-26,NaN
5,よふかしのうた,20巻,完結,2024-03-18,,NaN
6,ONE PIECE,108巻,109巻,2024-03-04,2024-07-05,NaN
7,ワンナイト・モーニング,10巻,11巻,2023-12-11,2024-04-22,NaN
8,喰う寝るふたり 住むふたり 続,5巻,完結,2023-08-19,,NaN
9,進撃のえろ子さん～変なお姉さんは男子高生と仲良くなりたい～,7巻,8巻,2024-02-29,2024-12-01,NaN
10,映像研には手を出すな!,8巻,9巻,2023-07-12,2024-07-11,NaN
